In [ ]:
import cv2
from flask import Flask, Response


app = Flask(__name__)


# Load the face detection model
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")


# Connect to the IP camera or Android IP Webcam
video_source = 'http://10.6.0.82/video'  # Replace with your stream URL
cap = cv2.VideoCapture(video_source)


def generate_frames():
   while True:
       success, frame = cap.read()
       if not success:
           break


       # Convert to grayscale
       gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)


       # Face detection
       faces = face_cascade.detectMultiScale(gray, 1.1, 4)
       for (x, y, w, h) in faces:
           cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)


       # Encode the frame as JPEG
       ret, buffer = cv2.imencode('.jpg', frame)
       frame = buffer.tobytes()


       # Yield frame in multipart format (for MJPEG)
       yield (b'--frame\r\n'
              b'Content-Type: image/jpeg\r\n\r\n' + frame + b'\r\n')


@app.route('/')
def index():
   return "<h1>Face Detection Stream</h1><img src='/video_feed'>"


@app.route('/video_feed')
def video_feed():
   return Response(generate_frames(),
                   mimetype='multipart/x-mixed-replace; boundary=frame')


if __name__ == "__main__":
   app.run(host="0.0.0.0", port=5004, debug=False)

